# Huấn luyện mô hình ngôn ngữ cho dữ liệu QA pháp luật

## Thư viện

In [6]:
# Colab / Kaggle
%pip install -U unsloth trl datasets evaluate rouge_score sacrebleu bert_score huggingface_hub python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 91.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 53.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 105.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 100.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 142.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]


## Constants và Configs

In [7]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset

# ==========================================
# Task / Model
# ==========================================
TASK_ID = "LEGAL_QUESTION_ANSWERING"
BASE_MODEL = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
HF_MODEL_REPO = "tmtdev/ViLegalQwen2.5-LegalQA-V2"
HF_DATASET_REPO = "tmtdev/ViLegal-QA-Dataset"

# ==========================================
# Sequence / QLoRA
# ==========================================
MAX_LENGTH = 2048
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
RANDOM_STATE = 42

# ==========================================
# Training
# ==========================================
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10

# Grounded QA: chỉ train các mẫu có relevant_laws
REQUIRE_LEGAL_CONTEXT = True

# Push / load
IS_PUSH_DATASET_TO_HF = False
LOAD_DATASET_FROM_HF = True
PUSH_MODEL_TO_HF = True

# Đánh giá generation - tăng lên None nếu muốn chạy toàn bộ test
EVAL_MAX_SAMPLES = 100

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


## Drive

In [8]:
# Nếu chạy Google Colab
from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = Path('/content')
DATA_DIR = Path('/content/drive/My Drive/legal-ai/data')
DATA_PATH = DATA_DIR / 'qa' / 'final_filtered_legal_qa_V2.jsonl'
OUTPUT_DIR = Path('/content/drive/My Drive/legal-ai/models/legal_qa_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.getenv('HF_TOKEN')

print('DATA_PATH:', DATA_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('HF_TOKEN:', 'Available' if HF_TOKEN else 'Not Available')


Mounted at /content/drive
DATA_PATH: /content/drive/My Drive/legal-ai/data/qa/final_filtered_legal_qa_V2.jsonl
OUTPUT_DIR: /content/drive/My Drive/legal-ai/models/legal_qa_v1
HF_TOKEN: Not Available


## Các utils

In [9]:
def load_jsonl_file(file_path: str | Path) -> list[dict]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    rows = []
    with file_path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
    return rows


def normalize_text(value) -> str:
    if value is None:
        return ''
    return str(value).strip()


def extract_citation_ids(text: str) -> list[int]:
    """Lấy marker [1], [2], ... theo thứ tự xuất hiện, không lặp."""
    ids = []
    for x in re.findall(r'\[(\d+)\]', text or ''):
        x = int(x)
        if x not in ids:
            ids.append(x)
    return ids


def format_legal_context(relevant_laws: list[dict]) -> str:
    """
    Biến relevant_laws thành context có ID trùng marker trong processed_answer.
    Không tự bổ sung căn cứ ngoài dữ liệu.
    """
    if not isinstance(relevant_laws, list):
        return ''

    blocks = []
    for law in relevant_laws:
        if not isinstance(law, dict):
            continue

        citation_id = law.get('citation_id')
        citation = normalize_text(law.get('citation'))
        content = normalize_text(law.get('content'))

        if citation_id is None or not citation:
            continue

        block = f"[{citation_id}] {citation}"
        if content:
            block += f"\nNội dung: {content}"
        blocks.append(block)

    return '\n\n'.join(blocks)


def validate_citation_alignment(processed_answer: str, relevant_laws: list[dict]) -> tuple[bool, str]:
    """Kiểm tra marker trong processed_answer có tồn tại trong relevant_laws hay không."""
    answer_ids = set(extract_citation_ids(processed_answer))
    law_ids = {
        int(x['citation_id'])
        for x in (relevant_laws or [])
        if isinstance(x, dict) and x.get('citation_id') is not None
    }

    missing = answer_ids - law_ids
    if missing:
        return False, f"Marker không có relevant_law tương ứng: {sorted(missing)}"
    return True, ''


## Dataset

### Tạo dataset

In [6]:
REQUIRED_SOURCE_COLUMNS = [
    'id', 'question', 'processed_answer', 'relevant_laws'
]


def build_qa_training_row(row: dict) -> dict | None:
    question = normalize_text(row.get('question'))
    answer = normalize_text(row.get('processed_answer'))
    relevant_laws = row.get('relevant_laws') or []

    if not question or not answer:
        return None

    legal_context = format_legal_context(relevant_laws)
    if REQUIRE_LEGAL_CONTEXT and not legal_context:
        return None

    ok, error = validate_citation_alignment(answer, relevant_laws)
    if not ok:
        # Không dùng sample lỗi citation làm ground truth train
        return None

    return {
        'id': row.get('id'),
        'task_id': TASK_ID,
        'input': question,
        'legal_context': legal_context,
        'output': answer,
        'title': row.get('title'),
        'original_output': row.get('answer'),
        'relevant_laws': relevant_laws,
        'source': row.get('source'),
        'topic': row.get('topic'),
        'url': row.get('url'),
    }


def create_dataset_from_source(rows: list[dict]) -> DatasetDict:
    print('Số dòng nguồn:', len(rows))

    missing_columns = [
        col for col in REQUIRED_SOURCE_COLUMNS
        if any(col not in row for row in rows[: min(50, len(rows))])
    ]
    if missing_columns:
        print('Warning - có thể thiếu columns:', missing_columns)

    prepared = []
    rejected = 0
    for row in rows:
        item = build_qa_training_row(row)
        if item is None:
            rejected += 1
            continue
        prepared.append(item)

    # chống duplicate question trước khi split
    df = pd.DataFrame(prepared)
    before = len(df)
    df = df.drop_duplicates(subset=['input'], keep='first').reset_index(drop=True)

    print('Prepared:', before)
    print('Rejected:', rejected)
    print('After duplicate removal:', len(df))

    if len(df) < 10:
        raise ValueError('Dataset quá nhỏ sau khi lọc.')

    dataset = Dataset.from_pandas(df, preserve_index=False)

    # 80 / 10 / 10
    first_split = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE)
    second_split = first_split['test'].train_test_split(test_size=0.5, seed=RANDOM_STATE)

    return DatasetDict({
        'train': first_split['train'],
        'validation': second_split['train'],
        'test': second_split['test'],
    })


if not LOAD_DATASET_FROM_HF:
    source_rows = load_jsonl_file(DATA_PATH)
    dataset = create_dataset_from_source(source_rows)

    train_data = dataset['train']
    val_data = dataset['validation']
    test_data = dataset['test']

    print(dataset)
    print('Sample:')
    print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


### Login HF

In [13]:
from huggingface_hub import login, notebook_login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    notebook_login()

### Upload dataset lên huggingface

In [7]:
from huggingface_hub import login, notebook_login

if IS_PUSH_DATASET_TO_HF:
    if HF_TOKEN:
        login(token=HF_TOKEN)
    else:
        notebook_login()

    dataset.push_to_hub(
        repo_id=HF_DATASET_REPO,
        private=False,
        token=HF_TOKEN,
    )
    print(f"Dataset pushed: https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print('Skip push dataset')


Skip push dataset


### Load dataset từ huggingface

In [8]:
if LOAD_DATASET_FROM_HF:
    dataset = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
    train_data = dataset['train']
    val_data = dataset['validation']
    test_data = dataset['test']

print('Train:', len(train_data))
print('Validation:', len(val_data))
print('Test:', len(test_data))


print('Sample:')
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


README.md:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.45MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  479kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  446kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/684 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/86 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/86 [00:00<?, ? examples/s]

Train: 684
Validation: 86
Test: 86
Sample:
{
  "id": "e17b190a0ef307fea16c",
  "task_id": "LEGAL_QUESTION_ANSWERING",
  "input": "Xin hỏi LuatVietnam: Tôi đang điều khiển ô tô con lưu thông trên cao tốc Hà Nội - Lào Cai thì gặp chướng ngại vật trước mặt. Khi tôi phanh xe để giảm tốc độ, bật xi nhan để đánh lái vào lề đường bên phải thì lập tức có một ô tô đi phía sau đâm vào đuôi xe, làm ô tô của tôi hỏng nặng. Sau đó liên tiếp 3 ô tô khác cũng “hít đuôi” nhau. Các bên không thỏa thuận được bồi thường thiệt hại như thế nào. Xin hỏi, trong trường hợp này lỗi thuộc về xe nào và cách giải quyết ra sao? Xin cảm ơn!",
  "legal_context": "[1] khoản 1 Điều 4 Thông tư 31/2019/TT-BGTVT\nNội dung: 1. Khi tham gia giao thông trên đường bộ, người điều khiển phương tiện phải nghiêm chỉnh chấp hành quy định về tốc độ, khoảng cách an toàn tối thiểu giữa hai xe (thuộc chủng loại xe cơ giới, xe máy chuyên dùng) được ghi trên biển báo hiệu đường bộ.\n\n[2] khoản 1 Điều 9 Thông tư 31/2019/TT-BGTVT\nNội d

## Cấu hình mô hình và huấn luyện
- Sử dụng unsloth cho nó nhẹ (hehehe)

In [9]:
class Prompt:
    SYSTEM_PROMPT = """
Bạn là model AI xử lý các nhiệm vụ pháp luật Việt Nam.
Thực hiện đúng TASK được chỉ định trong đầu vào.
Không tự bịa thông tin hoặc căn cứ pháp luật.
Chỉ sử dụng các căn cứ được cung cấp trong LEGAL_CONTEXT.
""".strip()

    LEGAL_QA_PROMPT = """
TASK: LEGAL_QUESTION_ANSWERING

Nhiệm vụ:
- Trả lời câu hỏi pháp luật Việt Nam dựa trên LEGAL_CONTEXT được cung cấp.
- Trả lời ngắn gọn, trực tiếp vào vấn đề.
- Giữ trích dẫn ngay tại nhận định mà căn cứ đó hỗ trợ.
- Marker trích dẫn phải dùng đúng citation_id: [1], [2], [3], ...
- Nếu một nhận định được hỗ trợ bởi nhiều căn cứ thì ghi riêng từng marker, ví dụ [1][2].
- Không gộp nhiều căn cứ thành một marker.
- Không sử dụng căn cứ không xuất hiện trong LEGAL_CONTEXT.
- Không chép lại dài dòng toàn bộ điều luật nếu không cần thiết.
""".strip()


def build_messages(task_id: str, input: str, legal_context: str):
    allowed_task_ids = [TASK_ID]
    if task_id not in allowed_task_ids:
        raise ValueError(
            f"Unknown task_id: {task_id}. Allowed task_ids: {allowed_task_ids}"
        )

    return [
        {
            'role': 'system',
            'content': Prompt.SYSTEM_PROMPT,
        },
        {
            'role': 'user',
            'content': (
                f"{Prompt.LEGAL_QA_PROMPT}\n\n"
                f"QUESTION:\n{input}\n\n"
                f"LEGAL_CONTEXT:\n{legal_context}"
            ),
        },
    ]


### Mô hình nền (Base model)

In [12]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LENGTH,
    dtype=None,                 # tự chọn fp16/bf16 phù hợp GPU
    load_in_4bit=LOAD_IN_4BIT, # QLoRA để giảm VRAM
)

tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = 'right'

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=RANDOM_STATE,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 04:29:52.827000 5620 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 04:29:53.052000 5620 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: ntphuc149/ViLegalQwen2.5-1.5B-Base had a bad pad_token (<|endoftext|>). Using pad_token = <|vision_pad|>.


Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [13]:
def format_training_example(example: dict) -> str:
    messages = build_messages(
        task_id=example['task_id'],
        input=example['input'],
        legal_context=example['legal_context'],
    )

    messages = messages + [
        {
            'role': 'assistant',
            'content': example['output'],
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def add_text_column(batch):
    texts = []
    n = len(batch['input'])
    for i in range(n):
        example = {k: batch[k][i] for k in batch.keys()}
        texts.append(format_training_example(example))
    return {'text': texts}


train_sft = train_data.map(add_text_column, batched=True)
val_sft = val_data.map(add_text_column, batched=True)
test_sft = test_data.map(add_text_column, batched=True)

print(train_sft[0]['text'])


Map:   0%|          | 0/684 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

<|im_start|>system
Bạn là model AI xử lý các nhiệm vụ pháp luật Việt Nam.
Thực hiện đúng TASK được chỉ định trong đầu vào.
Không tự bịa thông tin hoặc căn cứ pháp luật.
Chỉ sử dụng các căn cứ được cung cấp trong LEGAL_CONTEXT.<|im_end|>
<|im_start|>user
TASK: LEGAL_QUESTION_ANSWERING

Nhiệm vụ:
- Trả lời câu hỏi pháp luật Việt Nam dựa trên LEGAL_CONTEXT được cung cấp.
- Trả lời ngắn gọn, trực tiếp vào vấn đề.
- Giữ trích dẫn ngay tại nhận định mà căn cứ đó hỗ trợ.
- Marker trích dẫn phải dùng đúng citation_id: [1], [2], [3], ...
- Nếu một nhận định được hỗ trợ bởi nhiều căn cứ thì ghi riêng từng marker, ví dụ [1][2].
- Không gộp nhiều căn cứ thành một marker.
- Không sử dụng căn cứ không xuất hiện trong LEGAL_CONTEXT.
- Không chép lại dài dòng toàn bộ điều luật nếu không cần thiết.

QUESTION:
Xin hỏi LuatVietnam: Tôi đang điều khiển ô tô con lưu thông trên cao tốc Hà Nội - Lào Cai thì gặp chướng ngại vật trước mặt. Khi tôi phanh xe để giảm tốc độ, bật xi nhan để đánh lái vào lề đường b

### Cấu hình huấn luyện

In [14]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field='text',
    max_seq_length=MAX_LENGTH,

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    optim='adamw_8bit',
    lr_scheduler_type='linear',

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    logging_steps=LOGGING_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,

    report_to='none',
    seed=RANDOM_STATE,
    packing=False,
)


### Huấn luyện mô hình

In [15]:
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    args=training_args,
)

# Qwen2.5 dùng ChatML. Chỉ tính loss trên phần assistant response,
# tương đương ý tưởng labels=-100 cho prompt trong notebook 05.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Kiểm tra sample trước khi train
sample = trainer.train_dataset[0]
print('input_ids length:', len(sample['input_ids']))
if 'labels' in sample:
    active_labels = [x for x in sample['labels'] if x != -100]
    print('Assistant target:')
    print(tokenizer.decode(active_labels, skip_special_tokens=False))

train_result = trainer.train()
print(train_result)

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f'Saved adapter/tokenizer to: {OUTPUT_DIR}')


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/684 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/684 [00:00<?, ? examples/s]

Filter:   0%|          | 0/684 [00:00<?, ? examples/s]

Unsloth: Removed 3 out of 684 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


... (6 outputs left out)

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/My Drive/legal-ai/models/legal_qa_v1/checkpoint-43/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/My Drive/legal-ai/models/legal_qa_v1/checkpoint-86/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/My Drive/legal-ai/models/legal_qa_v1/checkpoint-129/tokenizer_config.json.


TrainOutput(global_step=129, training_loss=1.1308007572972498, metrics={'train_runtime': 1480.7866, 'train_samples_per_second': 1.38, 'train_steps_per_second': 0.087, 'total_flos': 1.8877266959698944e+16, 'train_loss': 1.1308007572972498, 'epoch': 3.0})


Unsloth: Restored added_tokens_decoder metadata in /content/drive/My Drive/legal-ai/models/legal_qa_v1/tokenizer_config.json.


Saved adapter/tokenizer to: /content/drive/My Drive/legal-ai/models/legal_qa_v1


### Upload mô hình lên huggingface

In [16]:
if PUSH_MODEL_TO_HF:
    if HF_TOKEN:
        login(token=HF_TOKEN)
    else:
        notebook_login()

    model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=False)
    tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=False)
    print(f"Model pushed: https://huggingface.co/{HF_MODEL_REPO}")
else:
    print('Skip push model')


README.md:   0%|          | 0.00/565 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

Saved model to https://huggingface.co/tmtdev/ViLegalQwen2.5-LegalQA-V2


... (2 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpyqmt72z8/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Model pushed: https://huggingface.co/tmtdev/ViLegalQwen2.5-LegalQA-V2


### Load lại model từ HF

In [10]:
import unsloth

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_REPO,  # "username/model-name"
    max_seq_length=MAX_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2279: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  f"not supported and can yield errors. Please verify that the checkpoint is compatible with the "
/usr/local/lib/python3.13/dist-packages/transformers/integrations/tensor_parallel.py:23: FutureWarning: `transformers.distributed.tensor_parallel.EpRouterParallel` is deprecated and will be removed in v5.23: import it from `transformers.distributed.expert_parallel` instead.
  from ..distributed.tensor_parallel import (
/usr/local/lib/python3.13/dist-packages/transformers/integrations/tensor_parallel.py:23: FutureWarning: `transformers.distributed.tensor_parallel.MoEParamShard` is deprecated and will be removed in v5.23: import it from `transformers.distributed.expert_parallel` instead.
  from ..distributed.tensor_parallel import (
/usr/local/lib/python

Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.22.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: ntphuc149/ViLegalQwen2.5-1.5B-Base had a bad pad_token (<|endoftext|>). Using pad_token = <|vision_pad|>.


Unsloth 2026.11.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Test thử mô hình

In [11]:
FastLanguageModel.for_inference(model)


def predict(
    input_text: str,
    relevant_laws: list[dict] | None = None,
    legal_context: str | None = None,
    task_id: str = TASK_ID,
    max_new_tokens: int = 512,
) -> str:
    if legal_context is None:
        legal_context = format_legal_context(relevant_laws or [])

    messages = build_messages(
        task_id=task_id,
        input=input_text,
        legal_context=legal_context,
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to(model.device)

    im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    input_len = inputs['input_ids'].shape[-1]
    return tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    ).strip()


sample = test_data[0]
pred = predict(
    input_text=sample['input'],
    legal_context=sample['legal_context'],
)

print('QUESTION:')
print(sample['input'])
print('\nEXPECTED:')
print(sample['output'])
print('\nPREDICTED:')
print(pred)


NameError: name 'test_data' is not defined

## Đánh giá mô hình
- Sử dụng các metrics của LLM NLP
  - BERTScore
  - ROUGE
  - ROUGE-L
  - ROUGE-1
  - ROUGE-2
  - BLEU

In [18]:
# Loss trên validation/test
val_metrics = trainer.evaluate(eval_dataset=val_sft, metric_key_prefix='val')
test_metrics = trainer.evaluate(eval_dataset=test_sft, metric_key_prefix='test')

print('Validation:', val_metrics)
print('Test:', test_metrics)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch
1.034293,1.928177,3


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch
1.034293,1.880968,3


Validation: {'val_loss': 1.9281772375106812}
Test: {'test_loss': 1.880967617034912}


In [4]:
%pip install -q  evaluate rouge-score sacrebleu bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import evaluate
from tqdm.auto import tqdm

rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')
bertscore_metric = evaluate.load('bertscore')



def run_generation_evaluation(test_dataset, max_samples=100):
    n = len(test_dataset) if max_samples is None else min(len(test_dataset), max_samples)
    subset = test_dataset.select(range(n))

    rows = []
    for example in tqdm(subset, desc='Generating'):
        prediction = predict(
            input_text=example['input'],
            legal_context=example['legal_context'],
            max_new_tokens=512,
        )

        row = {
            'id': example.get('id'),
            'input': example['input'],
            'reference': example['output'],
            'prediction': prediction,
        }
        rows.append(row)

    return pd.DataFrame(rows)


df_eval = run_generation_evaluation(test_data, EVAL_MAX_SAMPLES)

references = df_eval['reference'].tolist()
predictions = df_eval['prediction'].tolist()

rouge = rouge_metric.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False,
)

bleu = bleu_metric.compute(
    predictions=predictions,
    references=[[x] for x in references],
)

bertscore = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    model_type='xlm-roberta-base',
    lang='vi',
)

summary = {
    'rouge1': rouge.get('rouge1'),
    'rouge2': rouge.get('rouge2'),
    'rougeL': rouge.get('rougeL'),
    'bleu': bleu.get('score'),
    'bertscore_precision': float(np.mean(bertscore['precision'])),
    'bertscore_recall': float(np.mean(bertscore['recall'])),
    'bertscore_f1': float(np.mean(bertscore['f1'])),
}

print(json.dumps(summary, ensure_ascii=False, indent=2))
display(df_eval.head(10))


NameError: name 'test_data' is not defined

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

metrics_path = OUTPUT_DIR / f'legal_qa_metrics_{timestamp}.json'
predictions_path = OUTPUT_DIR / f'legal_qa_predictions_{timestamp}.csv'

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

df_eval.to_csv(predictions_path, index=False, encoding='utf-8-sig')

print('Metrics:', metrics_path)
print('Predictions:', predictions_path)
